# Fraud Detection Feature Engineering - Unified Environment

This notebook demonstrates feature engineering for fraud detection using EMR on EKS with NVIDIA RAPIDS acceleration.
It can run in any of the JupyterHub profiles: Data Processing, GPU Accelerated, or Unified.

## Features:
- EMR on EKS integration for Spark jobs
- NVIDIA RAPIDS GPU acceleration
- S3 data access with IRSA authentication
- Compatible with existing fraud detection pipeline

In [ ]:
import os
import boto3
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

## Environment Setup and Configuration

In [ ]:
# Get environment variables from JupyterHub configuration
VIRTUAL_CLUSTER_ID = os.environ.get('VIRTUAL_CLUSTER_ID', 'REPLACE_WITH_VIRTUAL_CLUSTER_ID')
EMR_EXECUTION_ROLE_ARN = os.environ.get('EMR_EXECUTION_ROLE_ARN', 'REPLACE_WITH_EMR_EXECUTION_ROLE_ARN')
S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', 'REPLACE_WITH_S3_BUCKET_NAME')
AWS_REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-west-2')

print("🔧 Environment Configuration:")
print(f"   Virtual Cluster ID: {VIRTUAL_CLUSTER_ID}")
print(f"   S3 Bucket: {S3_BUCKET_NAME}")
print(f"   Region: {AWS_REGION}")
print(f"   EMR Role: {EMR_EXECUTION_ROLE_ARN[:50]}..." if EMR_EXECUTION_ROLE_ARN else "   EMR Role: Not configured")

## Spark Session Configuration

Create a Spark session optimized for EMR on EKS with RAPIDS acceleration.

In [ ]:
def create_spark_session_for_eks():
    """Create Spark session configured for EMR on EKS with RAPIDS"""
    
    spark = SparkSession.builder \
        .appName("Fraud Detection - Feature Engineering") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.sql.shuffle.partitions", "200") \
        .config("spark.executor.memory", "4g") \
        .config("spark.executor.cores", "2") \
        .config("spark.executor.instances", "4") \
        .config("spark.rapids.sql.enabled", "true") \
        .config("spark.plugins", "com.nvidia.spark.SQLPlugin") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.WebIdentityTokenCredentialsProvider") \
        .getOrCreate()
    
    # Set log level to reduce noise
    spark.sparkContext.setLogLevel("WARN")
    
    return spark

# Create Spark session
print("🚀 Creating Spark session for EMR on EKS...")
spark = create_spark_session_for_eks()
print(f"✅ Spark session created: {spark.version}")
print(f"   Application ID: {spark.sparkContext.applicationId}")
print(f"   Master: {spark.sparkContext.master}")

## Data Loading from S3

Load the fraud detection datasets from S3 using the same data sources as the original EMR implementation.

In [ ]:
# Data paths - using the same structure as original implementation
# You can replace these with your actual S3 paths
CUSTOMERS_PATH = f"s3a://{S3_BUCKET_NAME}/customers/"
TERMINALS_PATH = f"s3a://{S3_BUCKET_NAME}/terminals/"
TRANSACTIONS_PATH = f"s3a://{S3_BUCKET_NAME}/transactions/"

print("📊 Loading datasets from S3...")
print(f"   Customers: {CUSTOMERS_PATH}")
print(f"   Terminals: {TERMINALS_PATH}")
print(f"   Transactions: {TRANSACTIONS_PATH}")

try:
    # Load customers data
    customers_df = spark.read.parquet(CUSTOMERS_PATH)
    print(f"✅ Customers loaded: {customers_df.count():,} records")
    
    # Load terminals data
    terminals_df = spark.read.parquet(TERMINALS_PATH)
    print(f"✅ Terminals loaded: {terminals_df.count():,} records")
    
    # Load transactions data
    transactions_df = spark.read.parquet(TRANSACTIONS_PATH)
    print(f"✅ Transactions loaded: {transactions_df.count():,} records")
    
except Exception as e:
    print(f"❌ Error loading data: {str(e)}")
    print("💡 Make sure your S3 bucket contains the fraud detection datasets")
    print("💡 Check that IRSA is properly configured for S3 access")

## Data Schema Exploration

In [ ]:
# Display schema information
print("📋 Dataset Schemas:")
print("\n🏪 Customers Schema:")
customers_df.printSchema()

print("\n🏧 Terminals Schema:")
terminals_df.printSchema()

print("\n💳 Transactions Schema:")
transactions_df.printSchema()

In [ ]:
# Show sample data
print("📊 Sample Data:")
print("\n🏪 Customers (first 5 rows):")
customers_df.show(5, truncate=False)

print("\n🏧 Terminals (first 5 rows):")
terminals_df.show(5, truncate=False)

print("\n💳 Transactions (first 5 rows):")
transactions_df.show(5, truncate=False)

## Feature Engineering Pipeline

Implement the same feature engineering logic as the original EMR implementation, optimized for RAPIDS acceleration.

In [ ]:
def create_time_features(df):
    """Create time-based features from transaction datetime"""
    
    return df.withColumn("yyyy", F.year("TX_DATETIME")) \
             .withColumn("mm", F.month("TX_DATETIME")) \
             .withColumn("dd", F.dayofmonth("TX_DATETIME")) \
             .withColumn("hour", F.hour("TX_DATETIME")) \
             .withColumn("minute", F.minute("TX_DATETIME")) \
             .withColumn("dayofweek", F.dayofweek("TX_DATETIME"))

def create_customer_features(df):
    """Create customer-based aggregation features"""
    
    # Define time windows for aggregations
    windows = {
        "15min": 15 * 60,  # 15 minutes in seconds
        "1hour": 60 * 60,  # 1 hour in seconds
        "1day": 24 * 60 * 60  # 1 day in seconds
    }
    
    # Convert TX_DATETIME to timestamp for window operations
    df_with_ts = df.withColumn("tx_timestamp", F.unix_timestamp("TX_DATETIME"))
    
    result_df = df_with_ts
    
    for window_name, window_seconds in windows.items():
        # Create window specification
        window_spec = Window.partitionBy("CUSTOMER_ID") \
                           .orderBy("tx_timestamp") \
                           .rangeBetween(-window_seconds, 0)
        
        # Calculate aggregations for this window
        result_df = result_df.withColumn(
            f"customer_id_nb_txns_{window_name}_window",
            F.count("TX_AMOUNT").over(window_spec)
        ).withColumn(
            f"customer_id_avg_amt_{window_name}_window",
            F.avg("TX_AMOUNT").over(window_spec)
        ).withColumn(
            f"customer_id_sum_amt_{window_name}_window",
            F.sum("TX_AMOUNT").over(window_spec)
        )
    
    return result_df.drop("tx_timestamp")

def create_terminal_features(df):
    """Create terminal-based aggregation features"""
    
    # Similar to customer features but partitioned by terminal
    windows = {
        "15min": 15 * 60,
        "1hour": 60 * 60,
        "1day": 24 * 60 * 60
    }
    
    df_with_ts = df.withColumn("tx_timestamp", F.unix_timestamp("TX_DATETIME"))
    result_df = df_with_ts
    
    for window_name, window_seconds in windows.items():
        window_spec = Window.partitionBy("TERMINAL_ID") \
                           .orderBy("tx_timestamp") \
                           .rangeBetween(-window_seconds, 0)
        
        result_df = result_df.withColumn(
            f"terminal_id_nb_txns_{window_name}_window",
            F.count("TX_AMOUNT").over(window_spec)
        ).withColumn(
            f"terminal_id_avg_amt_{window_name}_window",
            F.avg("TX_AMOUNT").over(window_spec)
        )
    
    return result_df.drop("tx_timestamp")

print("🔧 Feature engineering functions defined")

## Apply Feature Engineering

In [ ]:
print("⚙️ Starting feature engineering pipeline...")

# Step 1: Create time features
print("   📅 Creating time features...")
transactions_with_time = create_time_features(transactions_df)

# Step 2: Create customer aggregation features
print("   👤 Creating customer aggregation features...")
transactions_with_customer_features = create_customer_features(transactions_with_time)

# Step 3: Create terminal aggregation features
print("   🏧 Creating terminal aggregation features...")
transactions_with_all_features = create_terminal_features(transactions_with_customer_features)

# Step 4: Join with customer and terminal data
print("   🔗 Joining with customer and terminal data...")
final_features = transactions_with_all_features \
    .join(customers_df, "CUSTOMER_ID", "left") \
    .join(terminals_df, "TERMINAL_ID", "left")

print("✅ Feature engineering completed!")
print(f"   Final dataset shape: {final_features.count():,} rows, {len(final_features.columns)} columns")

## Feature Summary and Validation

In [ ]:
# Display final schema
print("📋 Final Feature Schema:")
final_features.printSchema()

# Show feature statistics
print("\n📊 Feature Statistics:")
final_features.describe().show()

# Check for fraud distribution
print("\n🚨 Fraud Distribution:")
fraud_distribution = final_features.groupBy("TX_FRAUD").count().collect()
for row in fraud_distribution:
    fraud_label = "Fraudulent" if row['TX_FRAUD'] == 1 else "Legitimate"
    percentage = (row['count'] / final_features.count()) * 100
    print(f"   {fraud_label}: {row['count']:,} ({percentage:.2f}%)")

## Save Processed Features to S3

In [ ]:
# Define output path
OUTPUT_PATH = f"s3a://{S3_BUCKET_NAME}/processed-features/"

print(f"💾 Saving processed features to: {OUTPUT_PATH}")

try:
    # Save as Parquet with partitioning for better performance
    final_features.write \
        .mode("overwrite") \
        .partitionBy("yyyy", "mm") \
        .parquet(OUTPUT_PATH)
    
    print("✅ Features saved successfully!")
    print(f"   Output location: {OUTPUT_PATH}")
    print(f"   Partitioned by: year (yyyy) and month (mm)")
    
except Exception as e:
    print(f"❌ Error saving features: {str(e)}")
    print("💡 Check S3 permissions and bucket configuration")

## Performance Metrics and Summary

In [ ]:
# Get Spark application metrics
print("📈 Spark Application Metrics:")
print(f"   Application ID: {spark.sparkContext.applicationId}")
print(f"   Application Name: {spark.sparkContext.appName}")
print(f"   Spark Version: {spark.version}")
print(f"   Default Parallelism: {spark.sparkContext.defaultParallelism}")

# Summary
print("\n🎯 Processing Summary:")
print(f"   ✅ Processed {final_features.count():,} transactions")
print(f"   ✅ Created {len(final_features.columns)} features")
print(f"   ✅ Data saved to S3: {OUTPUT_PATH}")
print(f"   ✅ Ready for ML training pipeline")

print("\n🚀 Next Steps:")
print("   1. Run the ML training notebook (02_fraud_detection_training_unified.ipynb)")
print("   2. Use Ray for distributed XGBoost training")
print("   3. Deploy model for inference")

## Cleanup

In [ ]:
# Stop Spark session
print("🧹 Cleaning up Spark session...")
spark.stop()
print("✅ Spark session stopped")